# ADS v2: Motor acquisition events

Builds 35 event families: `A0`, `A0.F1`, `A3.E1`, `A4a`, `A4b`, `A4.F1`, `A5a`, `A5.F1`, `A1`, `A1.E1`, `A2`, `A2.F1`, `A3`, `A3.F1`, `A3.B1.1`, `A2.E1`, `A5.1`, `A6`, `A6.F1`, `A6.B1.F1`, `A7.F1`, `A6.B1.1`, `A6.B1.4`, `A7.B1.4`, `A6.B1.2`, `A6.B1.3`, `A6.B1.6`, `A7`, `A6.B1.5`, `A6.B1.O1`, `A7.B1`, `A7.B1.1`, `A7.B1.2`, `A7.B1.3`, `A7.B1.5`.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("11_Motor_Acquisition", "started")


## A0: Straight into the CALL CENTRE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_inbound_sales_call                                                    */
/*  Description : Inbound conversations whose last inbound queue session is INBOUND_SALES_MOTOR, with t */
/*              : he customer's phone key.                                                              */
/*              :                                                                                       */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_inbound_sales_call as
select  c.ConversationId,
        c.SessionIndex,
        c.ConversationStartTime,
        case when c.OriginatingDirection = 'inbound' then c.AniPhoneKey
             when c.OriginatingDirection = 'outbound' then c.DnisPhoneKey end as CustomerPhoneKey
from    ${ads.ods}.CallSession c
join    (
        select  ConversationId, max(SessionIndex) as SessionIndex
        from    ${ads.ods}.CallSession
        where   QueueName is not null and OriginatingDirection = 'inbound' and ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        ) l     on l.ConversationId = c.ConversationId and l.SessionIndex = c.SessionIndex
where   c.QueueName = 'INBOUND_SALES_MOTOR'
and     c.ConversationStartTime >= timestamp '${ads.start_time}' and c.ConversationStartTime < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a0  (A0)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cells 6, 7, 9, 10, 72. Inbound calls whose last inbo */
/*              : und queue session is INBOUND_SALES_MOTOR and whose customer number matches no MFQ quo */
/*              : te mobile. Change: hash keys match +353 / 00353 / 08 forms the raw comparison missed, */
/*              :  and MFQ mobiles come from each quote's latest query (MFQ source bug rule), so expect */
/*              :  slightly fewer A0.                                                                   */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession, QuoteMotor                        */
/*              : Reads: stg.adsv2_motor_inbound_sales_call, ods_v2.QuoteMotor                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a0 as
select  xxhash64('A0', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A0'                                                as EventTypeId,
        'Straight into the CALL CENTRE' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(c.ConversationStartTime)                                    as EventDateTime,
                20                                                              as SourceSystemId,
                c.ConversationId                                                as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_inbound_sales_call c
        left join (
                select  distinct MobileKey
                from    ${ads.ods}.QuoteMotor
                where   MobileKey is not null and FirstCreatedDateTimeUtc >= timestamp '${ads.start_time}' and FirstCreatedDateTimeUtc < timestamp '${ads.end_time}'
                ) m     on m.MobileKey = c.CustomerPhoneKey
        where   m.MobileKey is null
        group by c.ConversationId, c.SessionIndex
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A0.F1: Call wait time-  or fails to make contact


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a0_f1  (A0.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cells 11, 73. Conversations on INBOUND_SALES_MOTO */
/*              : R that waited more than 600 seconds or were abandoned (D-E1.25; 04 took abandoned cal */
/*              : ls with any wait; wrap-up codes are not used).                                        */
/*              :                                                                                       */
/*              : Grain: Call   Product: Motor   Source: CallSession                                    */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a0_f1 as
select  xxhash64('A0.F1', 'Call',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A0.F1'                                             as EventTypeId,
        'Call wait time-  or fails to make contact' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Call'                                              as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                ConversationId                                                  as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.ods}.CallSession
        where   QueueName = 'INBOUND_SALES_MOTOR'
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by ConversationId
        having  sum(AcdWaitDurationMs) > 600000 or sum(Abandoned) > 0
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A3.E1: Outbound sales callback


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a3_e1  (A3.E1)                                                          */
/*  Description : Acquisition / Chill Effort. Notebook 04 cell 79. Outbound sales callbacks: outbound c */
/*              : onversations on OUTBOUND_SALES_MOTOR or INBOUND_SALES_MOTOR, one event per customer n */
/*              : umber at its first such call; the customer reference is the PhoneKey (dialled number) */
/*              : . Change (Vijay, 2026-10-19): either sales queue with OriginatingDirection 'outbound' */
/*              : , QD_* queues out of scope (04 looked for OUTBOUND_SALES_MOTOR on inbound calls).     */
/*              :                                                                                       */
/*              : Grain: Customer   Product: Motor   Source: CallSession                                */
/*              : Reads: ods_v2.CallSession                                                             */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a3_e1 as
select  xxhash64('A3.E1', 'Customer',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A3.E1'                                             as EventTypeId,
        'Outbound sales callback' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Customer'                                          as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        cast(null as string)                                         as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        cast(null as bigint)                                         as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(ConversationStartTime)                                      as EventDateTime,
                20                                                              as SourceSystemId,
                DnisPhoneKey                                                    as SourceCustomerReference,
                'PhoneKey'                                                      as SourceCustomerReferenceType
        from    ${ads.ods}.CallSession
        where   QueueName in ('OUTBOUND_SALES_MOTOR', 'INBOUND_SALES_MOTOR')
        and     OriginatingDirection = 'outbound'
        and     DnisPhoneKey is not null
        and     ConversationStartTime >= timestamp '${ads.start_time}' and ConversationStartTime < timestamp '${ads.end_time}'
        group by DnisPhoneKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A4a: Customer buys ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a4a  (A4a)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cells 17, 80. Active sales (Cat A1, OrgID 1, SaleCou */
/*              : nt 1, before the effective date) whose MFQ quote had a successful online payment in t */
/*              : he window; at the last such payment.                                                  */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: PolicySale, QuoteMotorPayment                 */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.QuoteMotorPayment                              */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a4a as
select  xxhash64('A4a', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A4a'                                               as EventTypeId,
        'Customer buys ONLINE' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  p.LastPaymentDateTime                                           as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                s.QuoteQueryGuid                                                as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                s.PolicyId,
                p.QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    (
                select  GuidKey, max(QuoteId) as QuoteId, max(DateUpdated) as LastPaymentDateTime
                from    ${ads.ods}.QuoteMotorPayment
                where   Status like '%|Success, code: 00%'
                and     PaymentStatus in (0, 5)
                and     DateUpdated >= timestamp '${ads.start_time}' and DateUpdated < timestamp '${ads.end_time}'
                group by GuidKey
                ) p     on p.GuidKey = s.QuoteQueryGuid
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A4b: Agent quotes and converts on the CALL


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a4b  (A4b)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 81. New-business motor sales with no MFQ quote  */
/*              : (agent quotes and converts on the call).                                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale                                   */
/*              : Reads: stg.adsv2_policies_sold                                                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a4b as
select  xxhash64('A4b', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A4b'                                               as EventTypeId,
        'Agent quotes and converts on the CALL' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(ReportingSaleDate as timestamp)                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold
        where   ReportingSaleType = 'New Business'
        and     PolicyTypeGroup = 'Motor'
        and     QuoteQueryGuid is null
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A4.F1: Web assist: starts online-  then calls to complete


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a4_f1  (A4.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 82. New-business motor sales with an MFQ quo */
/*              : te through Web Assist (starts online, calls to complete).                             */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, Quote                            */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.Quote                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a4_f1 as
select  xxhash64('A4.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A4.F1'                                             as EventTypeId,
        'Web assist: starts online-  then calls to complete' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(s.ReportingSaleDate as timestamp)                          as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                s.QuoteQueryGuid                                                as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                s.PolicyId,
                q.QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        left join ${ads.ods}.Quote q on q.SourceSystemId = 2 and q.SourceSystemQuoteReference = s.QuoteQueryGuid
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Motor'
        and     s.QuoteQueryGuid is not null
        and     s.Channel = 'b) Web Assist'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A5a: Payment attempted ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a5a  (A5a)                                                              */
/*  Description : Acquisition / Spine. Notebook 04 cell 83. Every MFQ online payment attempt in the win */
/*              : dow (one event per quote and minute).                                                 */
/*              :                                                                                       */
/*              : Grain: Payment Attempt   Product: Motor   Source: QuoteMotorPayment                   */
/*              : Reads: ods_v2.QuoteMotorPayment                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a5a as
select  xxhash64('A5a', 'Payment Attempt',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A5a'                                               as EventTypeId,
        'Payment attempted ONLINE' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Payment Attempt'                                   as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  DateUpdated                                                     as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.ods}.QuoteMotorPayment
        where   DateUpdated >= timestamp '${ads.start_time}' and DateUpdated < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A5.F1: Payment failed


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a5_f1  (A5.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 84. Failed MFQ online payments in the window */
/*              : : PaymentStatus 4 only (R-3, D-E1.2). 04 took every status but 0 / 5, so status 1 (st */
/*              : arted, no response: abandoned, not failed) was counted as a failure.                  */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotorPayment                             */
/*              : Reads: ods_v2.QuoteMotorPayment                                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a5_f1 as
select  xxhash64('A5.F1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A5.F1'                                             as EventTypeId,
        'Payment failed' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  DateUpdated                                                     as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.ods}.QuoteMotorPayment
        where   DateUpdated >= timestamp '${ads.start_time}' and DateUpdated < timestamp '${ads.end_time}'
        and     PaymentStatus = 4
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A1: Quote initiated ONLINE


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_mfq_quote_initiated                                                         */
/*  Description : MFQ quotes first created in the window (stg.A1_MotorAcquisitionQuoteInitiated): first */
/*              :  step 1 / step 5 times from the client steps, the retrieve token from the latest quer */
/*              : y row.                                                                                */
/*              :                                                                                       */
/*              : Reads: ods_v2.QuoteMotor, ods_v2.QuoteMotorStep                                       */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_mfq_quote_initiated as
select  q.GuidKey,
        q.QuoteId,
        q.FirstCreatedDateTimeUtc,
        q.RetrieveSessionToken,
        q.FirstStep1DateTimeUtc,
        s.FirstStep5DateTimeUtc
from    ${ads.ods}.QuoteMotor q
left join (
        select  GuidKey, min(StepDateTimeUtc) as FirstStep5DateTimeUtc
        from    ${ads.ods}.QuoteMotorStep
        where   StepId = 5
        group by GuidKey
        ) s     on s.GuidKey = q.GuidKey
where   q.FirstCreatedDateTimeUtc >= timestamp '${ads.start_time}' and q.FirstCreatedDateTimeUtc < timestamp '${ads.end_time}'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a1  (A1)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cells 8, 70. Every MFQ quote first created in the wi */
/*              : ndow, at its first creation. Change: times are UTC (ods_v2), the window applies to th */
/*              : e quote's first creation rather than to each query row.                               */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor, QuoteMotorStep                    */
/*              : Reads: stg.adsv2_mfq_quote_initiated                                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a1 as
select  xxhash64('A1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A1'                                                as EventTypeId,
        'Quote initiated ONLINE' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstCreatedDateTimeUtc                                         as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_quote_initiated
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A1.E1: Service retrieve


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a1_e1  (A1.E1)                                                          */
/*  Description : Acquisition / Chill Effort. Notebook 04 cell 71. No service grid data, so the same qu */
/*              : otes and times as A1 (as today).                                                      */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor                                    */
/*              : Reads: stg.adsv2_mfq_quote_initiated                                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a1_e1 as
select  xxhash64('A1.E1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A1.E1'                                             as EventTypeId,
        'Service retrieve' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstCreatedDateTimeUtc                                         as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_quote_initiated
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A2: Customer details complete


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a2  (A2)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cell 74. Quotes of A1 that reached step 5 (customer  */
/*              : details complete) in the window, at the first step 5.                                 */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor, QuoteMotorStep                    */
/*              : Reads: stg.adsv2_mfq_quote_initiated                                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a2 as
select  xxhash64('A2', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A2'                                                as EventTypeId,
        'Customer details complete' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstStep5DateTimeUtc                                           as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_quote_initiated
        where   FirstStep5DateTimeUtc >= timestamp '${ads.start_time}' and FirstStep5DateTimeUtc < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A2.F1: Drop off before completing the details


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a2_f1  (A2.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 75. Quotes of A1 that never reached step 5,  */
/*              : at their first creation.                                                              */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor, QuoteMotorStep                    */
/*              : Reads: stg.adsv2_mfq_quote_initiated                                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a2_f1 as
select  xxhash64('A2.F1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A2.F1'                                             as EventTypeId,
        'Drop off before completing the details' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstCreatedDateTimeUtc                                         as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                GuidKey                                                         as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_quote_initiated
        where   FirstStep5DateTimeUtc is null
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A3: Customer gets a price


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_mfq_first_quote                                                             */
/*  Description : A1 quotes with the SL_10 first-quote flag and insurers-quoted flag (replaces stg.A3_S */
/*              : L10_QQGUIDS), and whether it is a bot (the SL_10 view's ScraperFlag: Imperva, Chill I */
/*              : P rules, Fusion bot; R-4).                                                            */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_mfq_quote_initiated, ods_v2.QuoteMotorSummary, ods_v2.QuoteMotor     */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_mfq_first_quote as
select  a.GuidKey,
        a.QuoteId,
        a.FirstStep1DateTimeUtc,
        a.FirstStep5DateTimeUtc,
        a.RetrieveSessionToken,
        m.InsurerCount,
        s.InsurersQuoted,
        case when s.QuoteType in ('FQ', 'QQ') and s.FirstClickFlag = 'Y'
             then coalesce(s.QQ_All, 0) + coalesce(s.FQ_All, 0) else 0 end   as TotAllFirst,
        case when s.ScraperFlag is not null then 1 else 0 end           as BotSession
from    ${ads.stg}.adsv2_mfq_quote_initiated a
join    ${ads.ods}.QuoteMotorSummary s  on s.GuidKey = a.GuidKey
left join ${ads.ods}.QuoteMotor m       on m.GuidKey = a.GuidKey

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a3  (A3)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cells 13, 77. A first quote (SL_10 Tot-All-First = 1 */
/*              : ) where insurers quoted, not a retrieve, not a bot session, with a step 1; at the ste */
/*              : p 1 time. Change: stg.A3_SL10_QQGUIDS (a one-off copy of on-prem DWH.vw_rpt_SL_10_MFQ */
/*              : _Performance_Report, quotes 2024-01 .. 2026-09) is rebuilt from the view's own rule o */
/*              : n QuoteMotorSummary (edw.tbl_fact_mfq): Tot-All-First = QQ_All + FQ_All when QuoteTyp */
/*              : e in ('FQ','QQ') and FirstClickFlag = 'Y' (view line 471; 99.1% agreement with the co */
/*              : py, checked 2026-10-14), so A3 is no longer limited to 2024 onwards. 'Insurers quoted */
/*              : ' (cell 13) = at least one insurer priced (QuoteMotor.InsurerCount). Bots: the view's */
/*              :  ScraperFlag (R-4), not 04's MFQ_MON_Session check.                                   */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor, QuoteMotorSummary                 */
/*              : Reads: stg.adsv2_mfq_first_quote                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a3 as
select  xxhash64('A3', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A3'                                                as EventTypeId,
        'Customer gets a price' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstStep1DateTimeUtc                                           as EventDateTime,
                2                                                               as SourceSystemId,
                GuidKey                                                         as SourceQuoteReference,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_first_quote
        where   TotAllFirst = 1
        and     InsurersQuoted = 'Y'
        and     InsurerCount > 0
        and     BotSession = 0
        and     RetrieveSessionToken is null
        and     FirstStep1DateTimeUtc is not null
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A3.F1: No price returned-  told to call us


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a3_f1  (A3.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 78. A first quote (Tot-All-First = 1) where  */
/*              : no insurer quoted, that reached step 5; at the step 5 time.                           */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor, QuoteMotorSummary                 */
/*              : Reads: stg.adsv2_mfq_first_quote                                                      */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a3_f1 as
select  xxhash64('A3.F1', 'Quote',
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(cast(null as string) as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A3.F1'                                             as EventTypeId,
        'No price returned-  told to call us' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        cast(null as string)                                         as EventSourceReference,
        cast(null as string)                                         as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        cast(null as string)                                         as SourceCustomerReference,
        cast(null as string)                                         as SourceCustomerReferenceType,
        cast(null as bigint)                                         as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  FirstStep5DateTimeUtc                                           as EventDateTime,
                2                                                               as SourceSystemId,
                GuidKey                                                         as SourceQuoteReference,
                QuoteId
        from    ${ads.stg}.adsv2_mfq_first_quote
        where   TotAllFirst = 1
        and     InsurersQuoted = 'N'
        and     FirstStep5DateTimeUtc is not null
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A3.B1.1: Quote retrieved or iterated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a3_b1_1  (A3.B1.1)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 91. A quote of A1 that another quote of A1 ret */
/*              : rieved (its guid is the other's retrieve token), at its first creation. Change: the g */
/*              : uid goes in SourceQuoteReference (04 wrote it into SourcePolicyReference); a quote re */
/*              : trieved several times is one event (04 wrote one row per retrieving quote).           */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: QuoteMotor                                    */
/*              : Reads: stg.adsv2_mfq_quote_initiated                                                  */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a3_b1_1 as
select  xxhash64('A3.B1.1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A3.B1.1'                                           as EventTypeId,
        'Quote retrieved or iterated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  distinct
                a.FirstCreatedDateTimeUtc                                       as EventDateTime,
                2                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                cast(null as string)                                            as SourcePolicyReference,
                a.GuidKey                                                       as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                a.QuoteId
        from    ${ads.stg}.adsv2_mfq_quote_initiated a
        join    ${ads.stg}.adsv2_mfq_quote_initiated b  on b.RetrieveSessionToken = a.GuidKey
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A2.E1: This year retrieve (TYR)


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a2_e1  (A2.E1)                                                          */
/*  Description : Acquisition / Chill Effort. Notebook 04 cell 76. 'Motor % TYR %' emails sent in the w */
/*              : indow. Change: the customer reference is the email hash key (04 wrote the raw email), */
/*              :  source system 21 Xtremepush (04: 1), each Xtremepush event once (duplicate loads dro */
/*              : pped in ods_v2).                                                                      */
/*              :                                                                                       */
/*              : Grain: Quote   Product: Motor   Source: MessageEvent                                  */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a2_e1 as
select  xxhash64('A2.E1', 'Quote',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A2.E1'                                             as EventTypeId,
        'This year retrieve (TYR)' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Quote'                                             as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  EventDateTime                                                   as EventDateTime,
                21                                                              as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                QuoteGuidKey                                                    as SourceQuoteReference,
                EmailKey                                                        as SourceCustomerReference,
                case when EmailKey is not null then 'EmailKey' end              as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.ods}.MessageEvent
        where   EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        and     CampaignName like 'Motor % TYR %'
        and     InteractionType = 'sent'
        and     MessageType = 'EMAIL'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A5.1: Purchase confirmation sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a5_1  (A5.1)                                                            */
/*  Description : Acquisition / Spine. Notebook 04 cell 85. 'Online Motor Bind Confirmation' messages s */
/*              : ent in the window. Changes as A2.E1.                                                  */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: MessageEvent                                 */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a5_1 as
select  xxhash64('A5.1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A5.1'                                              as EventTypeId,
        'Purchase confirmation sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  EventDateTime                                                   as EventDateTime,
                21                                                              as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                QuoteGuidKey                                                    as SourceQuoteReference,
                EmailKey                                                        as SourceCustomerReference,
                case when EmailKey is not null then 'EmailKey' end              as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.ods}.MessageEvent
        where   EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
        and     CampaignName = 'Online Motor Bind Confirmation'
        and     InteractionType = 'sent'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6: Documents requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_motor_chase                                                                 */
/*  Description : Motor new-business chase snapshot rows with each row's own send date (DAY 1 / 7 / 14  */
/*              : / 20), the sale date, and whether any document (the 31 of A6) or one of the six (A6.B */
/*              : 1.4, A7.F1, A7.B1.4) is outstanding.                                                  */
/*              :                                                                                       */
/*              : Reads: ods_v2.DocumentChaseMotorVan                                                   */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_motor_chase as
select  PolicyCode,
        Campaign,
        cast(coalesce(try_cast(SendDateText as date),
                      try_cast(concat(right(SendDateText, 4), '-', substring(SendDateText, 4, 2), '-', left(SendDateText, 2)) as date))
             as timestamp)                                              as SendDate,
        try_cast(concat(right(SaleDate, 4), '-', substring(SaleDate, 4, 2), '-', left(SaleDate, 2)) as date) as SaleDate,
        AnyOutstanding,
        SixOutstanding
from    (
        select  PolicyCode,
                Campaign,
                SaleDate,
                case Campaign when 'DAY 1'  then Day1SendDate
                              when 'DAY 7'  then Day7SendDate
                              when 'DAY 14' then Day14SendDate
                              when 'DAY 20' then Day20SendDate end      as SendDateText,
                case when (Sign_Prop_Status = 'O' or Sign_Stat_Of_Fact_Status = 'O' or Comp_Fin_Agree_Status = 'O' or Comp_DDM_Status = 'O' or Proof_Of_NCB_Status = 'O' or Trans_Proof_Of_NCB_Status = 'O' or Ltr_Of_Driv_Exp_Status = 'O' or Prop_Driv_Lic_Status = 'O' or Nam_Driv_Lic_Status = 'O' or Gap_In_Cov_Ltr_Status = 'O' or SecondCar_Cert_Status = 'O' or Doc_Ltr_Status = 'O' or Engineers_Rpt_Status = 'O' or NCT_Status = 'O' or Veh_Lic_Cert_Status = 'O' or Irish_Reg_Status = 'O' or Main_Driver_Dec_Status = 'O' or Soc_Dom_Pleas_Dec_Status = 'O' or Comp_Car_Exper_Status = 'O' or DD_Conf_Ltr_Status = 'O' or Comp_Fin_Mand_Status = 'O' or Val_For_Spec_Item_Status = 'O' or Dri_Lic_Num_Status = 'O' or Comp_Lost_Cert_Dec_Status = 'O' or Orig_Cert_Status = 'O' or Cancel_Req_Status = 'O' or PPS_Num_Status = 'O' or Afford_State_Status = 'O' or Identification_Status = 'O' or Digital_Journey_Status = 'O' or Finance_Form_Status = 'O') then 1 else 0 end
                                                                        as AnyOutstanding,
                case when (Gap_In_Cov_Ltr_Status = 'O' or Val_For_Spec_Item_Status = 'O' or PPS_Num_Status = 'O' or Identification_Status = 'O' or Finance_Form_Status = 'O' or Digital_Journey_Status = 'O') then 1 else 0 end
                                                                        as SixOutstanding
        from    ${ads.ods}.DocumentChaseMotorVan
        where   PolicyTypeGroup = 'Motor'
        and     PolicyType = 'New Business'
        ) c

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6  (A6)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cell 86. New-business motor policies with any docume */
/*              : nt outstanding on the DAY 1 campaign, at the DAY 1 send date. Change: 04 (and v2 unti */
/*              : l this change) stamped it at the sale date; the second-car certificate is tested (04  */
/*              : compared a text literal, always false).                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: stg.adsv2_motor_chase                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6 as
select  xxhash64('A6', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6'                                                as EventTypeId,
        'Documents requested' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase
        where   Campaign = 'DAY 1'
        and     AnyOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.F1: Chases before the customer submits-  and documents never submitted


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_f1  (A6.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 87, corrected as Home HA6.F1 (D-E1.16, TO BE */
/*              :  CONFIRMED FOR MOTOR): one event per chase sent (DAY 1 / 7 / 14 / 20) while any docum */
/*              : ent was still outstanding and before the first accepted document, at its send date, s */
/*              : o the number of chases per policy is the count of these events. 04 kept one row per p */
/*              : olicy, DAY 20 only, at the sale date.                                                 */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, DocumentWorkflowStatus*/
/*              : Reads: stg.adsv2_motor_chase, ods_v2.DocumentWorkflowStatus                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_f1 as
select  xxhash64('A6.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.F1'                                             as EventTypeId,
        'Chases before the customer submits-  and documents never submitted' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  c.SendDate                                                      as EventDateTime,
                1                                                               as SourceSystemId,
                c.Campaign                                                      as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase c
        left join (
                select  PolicyCode, min(cast(EventTimestamp as timestamp)) as FirstAccepted
                from    ${ads.ods}.DocumentWorkflowStatus
                where   IsAccepted = true
                group by PolicyCode
                ) a     on a.PolicyCode = c.PolicyCode
        where   c.AnyOutstanding = 1
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        and     (a.FirstAccepted is null or c.SendDate < a.FirstAccepted)
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.F1: Motor Acquisition - Documents never submitted, policy at risk. Bites A5a customers hardest


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_f1  (A6.B1.F1)                                                    */
/*  Description : Acquisition / Friction. Notebook 04 cell 88, corrected (TO BE CONFIRMED): new-busines */
/*              : s motor policies that never submitted (no accepted document at all) and still had a d */
/*              : ocument outstanding at their last chase, at that chase's send date. 04 mirrored this  */
/*              : from Van VA6.B1.F1 and kept PolicyType 'Renewals', so it counted renewal policies as  */
/*              : acquisition friction, at the earliest sale date.                                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan, DocumentWorkflowStatus*/
/*              : Reads: stg.adsv2_motor_chase, ods_v2.DocumentWorkflowStatus                           */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_f1 as
select  xxhash64('A6.B1.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.F1'                                          as EventTypeId,
        'Motor Acquisition - Documents never submitted, policy at risk. Bites A5a customers hardest' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(c.SendDate)                                                 as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                c.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase c
        left join (
                select  distinct PolicyCode
                from    ${ads.ods}.DocumentWorkflowStatus
                where   IsAccepted = true
                ) a     on a.PolicyCode = c.PolicyCode
        where   c.AnyOutstanding = 1
        and     a.PolicyCode is null
        and     c.SendDate >= timestamp '${ads.start_time}' and c.SendDate < timestamp '${ads.end_time}'
        group by c.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.F1: Non-arrival-  and time taken to receive


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_f1  (A7.F1)                                                          */
/*  Description : Acquisition / Friction. Notebook 04 cell 90. New-business motor policies with one of  */
/*              : six documents outstanding on the DAY 20 campaign, at the DAY 20 send date (04 and v2  */
/*              : until this change: the sale date). OPEN: a document chase is not a certificate failin */
/*              : g to arrive (bugs and lessons, P9); only the time is fixed here.                      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: stg.adsv2_motor_chase                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_f1 as
select  xxhash64('A7.F1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.F1'                                             as EventTypeId,
        'Non-arrival-  and time taken to receive' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase
        where   Campaign = 'DAY 20'
        and     SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.1: Document request issued


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_1  (A6.B1.1)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 92, corrected as Home HA6.B1.1 (D-E1.16): the  */
/*              : DAY 1 chase send (request issued) for a new-business motor policy, at its send date.  */
/*              : 04 took every policy in the snapshot, at the sale date.                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: stg.adsv2_motor_chase                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_1 as
select  xxhash64('A6.B1.1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.1'                                           as EventTypeId,
        'Document request issued' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase
        where   Campaign = 'DAY 1'
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.4: Chase-  first reminder


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_4  (A6.B1.4)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 94, corrected as Home HA6.B1.4 (D-E1.16, TO BE */
/*              :  CONFIRMED FOR MOTOR): the first reminder is the DAY 7 chase send with one of six doc */
/*              : uments still outstanding, at its send date. 04 read DAY 1 at the sale date, close to  */
/*              : the rows of A6.                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: stg.adsv2_motor_chase                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_4 as
select  xxhash64('A6.B1.4', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.4'                                           as EventTypeId,
        'Chase-  first reminder' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  min(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase
        where   Campaign = 'DAY 7'
        and     SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1.4: Non-arrival chase-  inbound


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1_4  (A7.B1.4)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 103, corrected as Home HA7.B1.4 (D-E1.16): the */
/*              :  last chase send with one of six documents still outstanding (non-arrival chase), at  */
/*              : its send date. 04 stamped it at the sale date.                                        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: DocumentChaseMotorVan                        */
/*              : Reads: stg.adsv2_motor_chase                                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1_4 as
select  xxhash64('A7.B1.4', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1.4'                                           as EventTypeId,
        'Non-arrival chase-  inbound' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(SendDate)                                                   as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                PolicyCode                                                      as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                cast(null as bigint)                                            as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_motor_chase
        where   SixOutstanding = 1
        and     SendDate >= timestamp '${ads.start_time}' and SendDate < timestamp '${ads.end_time}'
        group by PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.2: Customer submits documents


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_2  (A6.B1.2)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 93. Document uploads on new-business motor sal */
/*              : es, in the window.                                                                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, DocumentUpload                   */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentUpload                                 */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_2 as
select  xxhash64('A6.B1.2', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.2'                                           as EventTypeId,
        'Customer submits documents' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(u.EventTimestamp as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                u.UploadId                                                      as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                s.PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentUpload u     on u.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Motor'
        and     cast(u.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(u.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.3: Documents validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_3  (A6.B1.3)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 96. Document workflow records (validation) on  */
/*              : new-business motor sales, in the window.                                              */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, DocumentWorkflowStatus           */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_3 as
select  xxhash64('A6.B1.3', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.3'                                           as EventTypeId,
        'Documents validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(w.EventTimestamp as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                w.WorkflowGuid                                                  as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                s.PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Motor'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.6: Documents received and validated


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_6  (A6.B1.6)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 98. As A6.B1.3, accepted documents only.       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, DocumentWorkflowStatus           */
/*              : Reads: stg.adsv2_policies_sold, ods_v2.DocumentWorkflowStatus                         */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_6 as
select  xxhash64('A6.B1.6', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.6'                                           as EventTypeId,
        'Documents received and validated' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  cast(w.EventTimestamp as timestamp)                             as EventDateTime,
                1                                                               as SourceSystemId,
                w.WorkflowGuid                                                  as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                s.PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.ods}.DocumentWorkflowStatus w on w.PolicyCode = s.PolicyCode
        where   s.ReportingSaleType = 'New Business'
        and     s.PolicyTypeGroup = 'Motor'
        and     cast(w.EventTimestamp as timestamp) >= timestamp '${ads.start_time}' and cast(w.EventTimestamp as timestamp) < timestamp '${ads.end_time}'
        and     w.IsAccepted = true
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7: Cert and disc sent


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7  (A7)                                                                */
/*  Description : Acquisition / Spine. Notebook 04 cell 89. Certificate transmitted / posted / transmis */
/*              : sion confirmed.                                                                       */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7 as
select  xxhash64('A7', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7'                                                as EventTypeId,
        'Cert and disc sent' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('New Business - Document Transmission Confirmed - Certificate',
                                       'New Business - Document Posted Confirmation - Certificate',
                                       'New Business - Document Transmitted - Certificate')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.5: Chase-  escalation


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_5  (A6.B1.5)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 95. A final chase document.                    */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_5 as
select  xxhash64('A6.B1.5', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.5'                                           as EventTypeId,
        'Chase-  escalation' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription like '%Chase%'
        and     d.EventDescription like '%Final%'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A6.B1.O1: Cancellation-  non-receipt


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a6_b1_o1  (A6.B1.O1)                                                    */
/*  Description : Acquisition / Outcome. Notebook 04 cell 97. An insurer-led cancellation email (regist */
/*              : ration) on a policy that also had a final chase. 'Insurer Led Cancelation' is Chill's */
/*              :  spelling, kept in ods_v2.                                                            */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a6_b1_o1 as
select  xxhash64('A6.B1.O1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A6.B1.O1'                                          as EventTypeId,
        'Cancellation-  non-receipt' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        left semi join (
                select  PolicyCode
                from    ${ads.stg}.adsv2_relay_document_event
                where   PolicyTypeGroup = 'Motor'
                and     EventDateTime >= timestamp '${ads.start_time}' and EventDateTime < timestamp '${ads.end_time}'
                and     EventDescription like '%Chase%'
                and     EventDescription like '%Final%'
                ) c     on c.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('Insurer Led Cancelation - Emailed Document - Reg canx email template',
                                       'Insurer Led Cancelation - Sent To Document Processing - Reg canx email template')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1: Cert and disc dispatch


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1  (A7.B1)                                                          */
/*  Description : Acquisition / Branch. Notebook 04 cell 99. Certificate transmitted (cert and disc dis */
/*              : patch).                                                                               */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1 as
select  xxhash64('A7.B1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1'                                             as EventTypeId,
        'Cert and disc dispatch' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription = 'New Business - Document Transmitted - Certificate'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1.1: Cert and disc dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1_1  (A7.B1.1)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 100. The same rule as A7.B1 (as today).        */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1_1 as
select  xxhash64('A7.B1.1', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1.1'                                           as EventTypeId,
        'Cert and disc dispatched' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription = 'New Business - Document Transmitted - Certificate'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1.2: Document pack dispatched


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1_2  (A7.B1.2)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 101. Terms of Business transmitted or emailed. */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1_2 as
select  xxhash64('A7.B1.2', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1.2'                                           as EventTypeId,
        'Document pack dispatched' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Emailed Document - Terms Of Business')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1.3: Customer receipt-  implicit


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1_3  (A7.B1.3)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 102. The same rule as A7.B1.2 (as today).      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1_3 as
select  xxhash64('A7.B1.3', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1.3'                                           as EventTypeId,
        'Customer receipt-  implicit' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription in ('New Business - Document Transmitted - Terms Of Business',
                                       'New Business - Emailed Document - Terms Of Business')
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

## A7.B1.5: Replacement requested


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_evt_a7_b1_5  (A7.B1.5)                                                      */
/*  Description : Acquisition / Branch. Notebook 04 cell 104. A duplicate certificate transmitted.      */
/*              :                                                                                       */
/*              : Grain: Policy   Product: Motor   Source: PolicySale, PolicyDocumentEvent              */
/*              : Reads: stg.adsv2_policies_sold, stg.adsv2_relay_document_event                        */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_evt_a7_b1_5 as
select  xxhash64('A7.B1.5', 'Policy',
                 coalesce(cast(x.EventSourceReference as string), ''),
                 coalesce(cast(x.SourcePolicyReference as string), ''),
                 coalesce(cast(x.SourceQuoteReference as string), ''),
                 coalesce(cast(x.SourceCustomerReference as string), ''),
                 cast(date_trunc('MINUTE', x.EventDateTime) as string))          as EventId,
        'A7.B1.5'                                           as EventTypeId,
        'Replacement requested' as EventDescription,
        'Motor'                                             as PolicyTypeGroup,
        'Policy'                                            as Grain,
        date_trunc('MINUTE', x.EventDateTime)                       as EventDateTime,
        cast(x.EventDateTime as date)                               as EventDate,
        'actual'                                            as EventTimeBasis,
        x.SourceSystemId                                             as SourceSystemId,
        x.EventSourceReference                                       as EventSourceReference,
        x.SourcePolicyReference                                      as SourcePolicyReference,
        x.SourceQuoteReference                                       as SourceQuoteReference,
        x.SourceCustomerReference                                    as SourceCustomerReference,
        x.SourceCustomerReferenceType                                as SourceCustomerReferenceType,
        x.PolicyId                                                   as PolicyId,
        x.QuoteId                                                    as QuoteId,
        '${ads.batch_id}'                                         as AdsBatchId
from    (
        select  max(d.EventDateTime)                                            as EventDateTime,
                1                                                               as SourceSystemId,
                cast(null as string)                                            as EventSourceReference,
                s.PolicyCode                                                    as SourcePolicyReference,
                cast(null as string)                                            as SourceQuoteReference,
                cast(null as string)                                            as SourceCustomerReference,
                cast(null as string)                                            as SourceCustomerReferenceType,
                max(s.PolicyId)                                                 as PolicyId,
                cast(null as bigint)                                            as QuoteId
        from    ${ads.stg}.adsv2_policies_sold s
        join    ${ads.stg}.adsv2_relay_document_event d on d.PolicyCode = s.PolicyCode
        where   d.PolicyTypeGroup = 'Motor'
        and     d.EventDateTime >= timestamp '${ads.start_time}' and d.EventDateTime < timestamp '${ads.end_time}'
        and     s.ReportingSaleType = 'New Business'
        and     d.EventDescription = 'Duplicate Certificate - Document Transmitted - Certificate'
        group by s.PolicyCode
        ) x
where   x.EventDateTime is not null
and     x.EventDateTime >= add_months(timestamp '${ads.start_time}', -1)     -- no event more than a month
and     x.EventDateTime <  add_months(timestamp '${ads.end_time}', 1)        -- either side of the window

In [ ]:
%%pyspark
# Events per family (a family with no events is reported, not failed: check its source and window)
for t in ["adsv2_evt_a0", "adsv2_evt_a0_f1", "adsv2_evt_a3_e1", "adsv2_evt_a4a", "adsv2_evt_a4b", "adsv2_evt_a4_f1", "adsv2_evt_a5a", "adsv2_evt_a5_f1", "adsv2_evt_a1", "adsv2_evt_a1_e1", "adsv2_evt_a2", "adsv2_evt_a2_f1", "adsv2_evt_a3", "adsv2_evt_a3_f1", "adsv2_evt_a3_b1_1", "adsv2_evt_a2_e1", "adsv2_evt_a5_1", "adsv2_evt_a6", "adsv2_evt_a6_f1", "adsv2_evt_a6_b1_f1", "adsv2_evt_a7_f1", "adsv2_evt_a6_b1_1", "adsv2_evt_a6_b1_4", "adsv2_evt_a7_b1_4", "adsv2_evt_a6_b1_2", "adsv2_evt_a6_b1_3", "adsv2_evt_a6_b1_6", "adsv2_evt_a7", "adsv2_evt_a6_b1_5", "adsv2_evt_a6_b1_o1", "adsv2_evt_a7_b1", "adsv2_evt_a7_b1_1", "adsv2_evt_a7_b1_2", "adsv2_evt_a7_b1_3", "adsv2_evt_a7_b1_5"]:
    n = spark.table(f"{stg_schema}.{t}").count()
    print(f"{t:<40} {n:>12,}" + ("   <- no events" if n == 0 else ""))
ads_run_log("11_Motor_Acquisition", "succeeded")
